# Latent Diffusion (Stable Diffusion) — High-Resolution Image Synthesis with Latent Diffusion Models (2021)

_Papers · Generative Models_

**Run the diffusion model inside a compressed autoencoder latent instead of on raw pixels — far cheaper — and steer it with text through cross-attention.**

---

This notebook is a practice scaffold for the **Latent Diffusion (Stable Diffusion) — High-Resolution Image Synthesis with Latent Diffusion Models (2021)** lesson. We measure the per-step saving of diffusing in a compressed latent, train a tiny autoencoder + latent diffusion model end to end, sample from it, and wire up cross-attention text conditioning. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Measure the per-step saving (and define the denoiser)

The core idea of latent diffusion is to run the expensive denoiser on a small compressed latent instead of raw pixels. A conv U-Net's work scales with the number of spatial positions, so downsampling by a factor `f` cuts positions — and cost — by about `f^2`. Here we count positions to predict the saving, define a tiny conv denoiser, and actually time one step at pixel vs latent resolution to confirm the ~`f^2` speed-up (conv overheads pull the measured ratio slightly below the ideal).

In [ ]:
import torch
import torch.nn as nn
import math
import time

torch.manual_seed(0)

T = 50

# --- 0. Worked example: the f^2 per-step saving (positions pixel vs latent). ---
def positions(H, f):                       # spatial positions a conv U-Net processes per step
    return (H // f) ** 2

for f in (4, 8):
    pix = positions(256, 1)
    lat = positions(256, f)
    ratio = pix // lat
    print(f"f={f}: pixel {pix} positions, latent {lat}, ratio {ratio} (= f^2 = {f*f})")
# f=4: ratio 16 ;  f=8: ratio 64

# --- A toy conv 'U-Net'-ish denoiser (same per-position work at any resolution). ---
class ConvDenoiser(nn.Module):
    def __init__(self, ch):                # ch = number of channels it denoises
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch + 1, 64, 3, padding=1), nn.SiLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.SiLU(),
            nn.Conv2d(64, ch, 3, padding=1))

    def forward(self, x, t):               # t broadcast as an extra channel (time conditioning)
        tc = (t.float() / T).view(-1, 1, 1, 1).expand(x.size(0), 1, x.size(2), x.size(3))
        return self.net(torch.cat([x, tc], 1))

# Time one step at pixel resolution (3ch, 64x64) vs latent (4ch, 16x16), f=4. -------
def time_step(ch, H, iters=40):
    net = ConvDenoiser(ch)
    x = torch.randn(8, ch, H, H)
    tb = torch.randint(0, 50, (8,))
    for _ in range(5):                     # warm up
        net(x, tb)
    s = time.time()
    for _ in range(iters):
        net(x, tb)
    return (time.time() - s) / iters

t_pix = time_step(3, 64)
t_lat = time_step(4, 16)
print(f"\none step  pixel(3x64x64) {t_pix*1e3:.2f} ms   latent(4x16x16) {t_lat*1e3:.2f} ms"
      f"   speed-up {t_pix / t_lat:.1f}x  (expect ~f^2=16x; conv overheads pull it toward that)")


### Step 2 — Train and freeze the autoencoder (Stage 1)

Latent diffusion is a two-stage recipe. First we train a small autoencoder: the encoder downsamples a 32×32 image to a 4×8×8 latent (`f=4`), the decoder reconstructs it. We train on smooth low-rank toy "images" a tiny AE can reproduce, then **freeze** the encoder and decoder — exactly as the paper trains the perceptual autoencoder once and reuses it.

In [ ]:
# --- 1. A tiny frozen-style autoencoder: encoder downsamples by f=4, decoder upsamples. ---
class Encoder(nn.Module):                  # 3x32x32 image -> 4x8x8 latent (f=4)
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(3, 32, 4, 2, 1), nn.SiLU(),   # 32 -> 16
                                 nn.Conv2d(32, 64, 4, 2, 1), nn.SiLU(),  # 16 -> 8
                                 nn.Conv2d(64, 4, 3, 1, 1))              # latent channels = 4

    def forward(self, x):
        return self.net(x)

class Decoder(nn.Module):                  # 4x8x8 latent -> 3x32x32 image
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.ConvTranspose2d(4, 64, 4, 2, 1), nn.SiLU(),  # 8 -> 16
                                 nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.SiLU(), # 16 -> 32
                                 nn.Conv2d(32, 3, 3, 1, 1))

    def forward(self, z):
        return self.net(z)

# Toy 'images': smooth low-rank patterns so a small AE can reconstruct them. -------
def sample_images(n):
    a = torch.randn(n, 1, 1, 1)
    b = torch.randn(n, 1, 1, 1)
    gx = torch.linspace(-1, 1, 32).view(1, 1, 1, 32)
    gy = torch.linspace(-1, 1, 32).view(1, 1, 32, 1)
    img = torch.tanh(a * gx + b * gy)            # one smooth gradient per sample
    return img.repeat(1, 3, 1, 1)                # 3 channels

enc, dec = Encoder(), Decoder()
ae_opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=2e-3)
for step in range(400):                          # Stage 1: train the autoencoder, then freeze
    x = sample_images(128)
    xr = dec(enc(x))
    ae_loss = ((x - xr) ** 2).mean()
    ae_opt.zero_grad()
    ae_loss.backward()
    ae_opt.step()
for p in list(enc.parameters()) + list(dec.parameters()):
    p.requires_grad_(False)
print(f"\nautoencoder recon MSE (frozen): {ae_loss.item():.4f}   latent shape {tuple(enc(sample_images(1)).shape)}")


### Step 3 — Train the diffusion model *in the latent* (Stage 2)

Now Stage 2: a standard DDPM, but the forward-noising and denoising happen on the encoded 4-channel latent `z`, not on pixels. We build the noise schedule, encode each image (no gradient flows into the frozen AE), add noise at a random timestep, and train the denoiser to predict that noise — this is the `L_LDM` objective (Eq. 2). Because the latent is 16× smaller, each step is cheap.

In [ ]:
# --- 2. DDPM schedule + Stage 2: diffusion IN the latent (Eq. 2, L_LDM). ---
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
abar = torch.cumprod(alphas, 0)
unet = ConvDenoiser(4)                            # denoises 4-channel latents
opt = torch.optim.Adam(unet.parameters(), lr=2e-3)
for step in range(1500):
    x = sample_images(128)
    z = enc(x)                                    # encode to latent (no grad into AE)
    tb = torch.randint(0, T, (128,))
    eps = torch.randn_like(z)
    ab = abar[tb].view(-1, 1, 1, 1)
    zt = ab.sqrt() * z + (1 - ab).sqrt() * eps    # forward-noise the LATENT (DDPM Eq. 4 on z)
    loss = ((eps - unet(zt, tb)) ** 2).mean()     # Eq. 2: L_LDM (predict the noise, in latent)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 500 == 0:
        print(f"  L_LDM step {step:4d}  loss {loss.item():.4f}")


### Step 4 — Sample, run the pixel-space ablation, and add cross-attention

Three things tie it together. (1) We sample by running the reverse DDPM chain *in latent space*, then decode the clean latent to an image **once** at the end — the decode is the only pixel-resolution op. (2) An ablation times one step in pixel vs latent space to re-confirm the latent is far cheaper. (3) We build the cross-attention block (Sec 3.3): the image latent supplies the queries while the encoded condition (a text prompt) supplies the keys and values, so each image location pulls in the prompt content relevant to it.

In [ ]:
# --- 3. Sample a latent (DDPM Alg. 2), then DECODE ONCE to an image. ---
@torch.no_grad()
def sample(n=64):
    z = torch.randn(n, 4, 8, 8)                   # start: pure-noise LATENT
    for t in reversed(range(T)):
        tb = torch.full((n,), t, dtype=torch.long)
        e = unet(z, tb)
        a, ab = alphas[t], abar[t]
        mean = (1 / a.sqrt()) * (z - ((1 - a) / (1 - ab).sqrt()) * e)
        if t > 0:
            z = mean + betas[t].sqrt() * torch.randn_like(z)
        else:
            z = mean
    return dec(z)                                 # decode the clean latent ONCE -> image

imgs = sample()
real = sample_images(64)
# crude quality proxy: do generated images match the real images' value statistics?
print(f"\nsamples mean {imgs.mean():.3f} (real {real.mean():.3f})  "
      f"std {imgs.std():.3f} (real {real.std():.3f})")
# Our small run -- not the paper's reported number.

# --- 4. ABLATION: same denoiser, but diffuse in PIXEL space (f=1). Per step is ~f^2 costlier. ---
print("\nABLATION: one step pixel-space (f=1) vs latent (f=4):")
print(f"  pixel  step {time_step(3, 32)*1e3:.2f} ms over 32x32")
print(f"  latent step {time_step(4, 8)*1e3:.2f} ms over 8x8  -> latent is much cheaper per step")

# --- 5. Cross-attention conditioning block (Sec 3.3): image latent = Q, condition = K,V. ---
class CrossAttn(nn.Module):
    def __init__(self, d_img, d_ctx, d=64):
        super().__init__()
        self.Wq = nn.Linear(d_img, d)
        self.Wk = nn.Linear(d_ctx, d)
        self.Wv = nn.Linear(d_ctx, d)
        self.d = d

    def forward(self, phi, tau):                  # phi: image tokens (N,L,d_img); tau: cond (N,M,d_ctx)
        Q = self.Wq(phi)                          # Q from image
        K = self.Wk(tau)                          # K from condition
        V = self.Wv(tau)                          # V from condition
        att = torch.softmax(Q @ K.transpose(1, 2) / math.sqrt(self.d), -1)   # softmax(QK^T/sqrt d)
        return att @ V                                            # Attention(Q,K,V) (Sec 3.3)

ca = CrossAttn(d_img=4, d_ctx=16)
phi = torch.randn(2, 64, 4)        # flattened 8x8 latent feature map (N=64 positions)
tau = torch.randn(2, 7, 16)        # tau_theta(y): 7 'prompt token' vectors
print("\ncross-attention output shape:", tuple(ca(phi, tau).shape), "(image latent attends to the prompt)")


## Visualize the data & results

_How does the per-step denoising cost fall as the autoencoder downsamples harder (factor f), and is the saving really f^2?_

### Step 1 — Set up the timing experiment

To see the saving as a curve, we redefine the same conv denoiser and a `time_step` helper that times one forward pass at a given spatial size. We fix a 64×64 pixel-space baseline (`f=1`) and time it once; the next step sweeps the downsampling factor and compares against this baseline.

In [ ]:
import torch
import torch.nn as nn
import time

torch.manual_seed(0)
T = 50


class ConvDenoiser(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(ch + 1, 64, 3, padding=1), nn.SiLU(),
                                 nn.Conv2d(64, 64, 3, padding=1), nn.SiLU(),
                                 nn.Conv2d(64, ch, 3, padding=1))

    def forward(self, x, t):
        tc = (t.float() / T).view(-1, 1, 1, 1).expand(x.size(0), 1, x.size(2), x.size(3))
        return self.net(torch.cat([x, tc], 1))


def time_step(H, ch=4, iters=50):
    net = ConvDenoiser(ch)
    x = torch.randn(8, ch, H, H)
    tb = torch.randint(0, T, (8,))
    for _ in range(5):
        net(x, tb)
    s = time.time()
    for _ in range(iters):
        net(x, tb)
    return (time.time() - s) / iters

base_H = 64                                    # pixel-space resolution (f=1)
t0 = time_step(base_H)


### Step 2 — Sweep the downsampling factor and check the f^2 law

Now we sweep `f` over 1, 2, 4, 8, 16. For each we shrink the side by `f`, time one step, and print the measured time ratio against the ideal `1/f^2`. The measured column tracks `1/f^2` closely, flattening a little at large `f` as fixed conv overheads start to dominate — confirming the per-step cost really does fall like the square of the downsampling factor.

In [ ]:
print("f   positions   1/f^2     measured/t0")
for f in (1, 2, 4, 8, 16):
    H = max(base_H // f, 1)
    t = time_step(H)
    print(f"{f:<3} {H*H:<11} {1/f**2:<9.4f} {t / t0:.4f}")
# Positions = (256/f)^2 on a 256-image; measured ratio tracks 1/f^2, flattening slightly at large f.


## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You train the denoiser in an $f=4$ latent and it generates fine. Now train an
            identical-budget denoiser in pixel space ($f=1$, no encoder) on the same toy images. What do
            you expect for (a) the per-step cost and (b) sample quality at a fixed wall-clock / step budget, and
            what does the comparison demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Encode once with $f=4$, then count positions: a $32\times32$ toy image has $1024$ pixel positions but only $8\times8=64$ latent positions. — _The denoiser's work scales with positions; $1024/64 = 16 = f^2$ confirms the per-step saving._
- Train both denoisers for the same number of steps and time one forward pass of each. — _An honest ablation changes exactly one thing &mdash; where the diffusion runs (pixel vs latent) &mdash; holding the loss, optimizer, and step count fixed._
- At a fixed compute budget the latent model fits more, faster updates and reaches good samples sooner; the pixel model is far slower per step. — _This mirrors the paper's &sect;4.1 finding that LDM-1 (pixel) trains slowly and lags LDM-8 by a large FID gap._

**Answer:** (a) The pixel-space step is about $f^2=16\times$ more expensive per step (it processes $16\times$
                 more positions). (b) At a fixed budget the latent model trains faster and reaches good samples
                 sooner; the pixel model lags. This isolates the contribution: the compression-then-diffuse
                 design, not a fancier denoiser, is what buys the speed &mdash; matching the paper's reported FID
                 gap of 38 between pixel-based LDM-1 and LDM-8. (The CODE includes this ablation.)

</details>

**Problem 2.** On a $512\times512$ image, how many fewer spatial positions does the U-Net process per step at $f=8$
            versus pixel space, and what is the per-step speed-up?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Pixel positions: $512\times512 = 262{,}144$. — _One position per pixel in pixel-space diffusion._
- Latent spatial size at $f=8$: $512/8 = 64$, so $64\times64 = 4{,}096$ positions. — _$f$ divides each spatial dimension; the latent is $64\times64$._
- Ratio: $262{,}144 / 4{,}096 = 64$. — _And $64 = f^2 = 8^2$ &mdash; the saving is always $f^2$._

**Answer:** $262{,}144 \to 4{,}096$ positions, a $64\times$ reduction &mdash; exactly
                 $f^2 = 8^2 = 64$. Each denoising step is about $64\times$ cheaper. (More aggressive than $f=4$'s
                 $16\times$, but the paper cautions that pushing $f$ too high starts to lose image detail.)

</details>

**Problem 3.** In the cross-attention block, the queries are $Q=W_Q^{(i)}\varphi_i(z_t)$ and the keys/values are
            $K=W_K^{(i)}\tau_\theta(y)$, $V=W_V^{(i)}\tau_\theta(y)$. Why this assignment, and what would break
            if you swapped which side gives queries vs keys/values?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read $\text{Attention}(Q,K,V)=\text{softmax}(QK^\top/\sqrt{d})V$: each query row picks a weighted blend of the value rows, weighted by query-key similarity. — _The query decides "what am I looking for"; keys/values are "what is on offer"._
- Here the image latent $\varphi_i(z_t)$ should pull in relevant pieces of the prompt $\tau_\theta(y)$, so the image is the query and the text is keys/values. — _Each spatial location of the image fetches the prompt content relevant to it &mdash; the prompt conditions the image._
- Swap them and the text would attend to the image instead, producing text-shaped outputs and never injecting the prompt into the latent. — _Attention output has the shape of the query side; you would denoise without using the condition._

**Answer:** The image latent supplies queries because it is the thing being generated and must look up
                 relevant prompt content; the encoded condition $\tau_\theta(y)$ supplies keys and values because
                 it is the information being injected. Swapping makes the prompt attend to the image &mdash; the
                 output takes the prompt's shape and the condition never enters the denoiser, so text conditioning
                 silently fails.

</details>